In [1]:
'''
task: find out if 2 functions are uncorrelated and or independent
'''

%run -i ../3.expectations-and-moments/explore.ipynb
%matplotlib widget

In [2]:
#use plotly for smoother interactions
import plotly.graph_objects as go

def _plot_function(f, a=-10, b=10, n=4096):
  xs = [a + i * (b - a) / n for i in range(n + 1)]
  ys = [f(x) for x in xs]
  fig = go.Figure(data=[go.Scatter(x=xs, y=ys, mode='lines')])
  fig.add_hline(y=0, line_color='black', line_width=1)
  fig.add_vline(x=0, line_color='black', line_width=1)
  fig.show()
  return fig

def _plot_surface(g, x_lab='x', y_lab='y', z_lab='posterior', ax_lo=-5, ax_hi=5, ay_lo=-5, ay_hi=5, n=60):
  xs = np.linspace(ax_lo, ax_hi, n + 1)
  ys = np.linspace(ay_lo, ay_hi, n + 1)
  Z = [[g(x, y) for x in xs] for y in ys]
  fig = go.Figure(data=[go.Surface(x=xs, y=ys, z=Z)])
  fig.update_layout(scene=dict(xaxis_title=x_lab, yaxis_title=y_lab, zaxis_title=z_lab), margin=dict(l=0, r=0, t=30, b=0),)
  fig.show()

In [3]:
# _marginalize(p(x, y), y) -> p(y)
# py() = integral(p(x, y), dx)
# integral(p(x, y), dx) => as x changes when y doesnt
def _marginalize(parent, part):
  if part == 'y':
    def py(y):
      def pxydx(x):
        return parent(x, y)
      return _simpson_integrate(pxydx)
    return py
  elif part == 'x':
    def px(x):
      def pxydy(y):
        return parent(x, y)
      return _simpson_integrate(pxydy)
    return px


In [4]:
if __name__ == '__main__' and '__file__' not in globals():
  def make_p1xy():
    def p1xy(x, y):
      px = _make_gaussian(0.0, 0.5)
      py = _make_gaussian(2.0, 3.5)
      return px(x) * py(y)
    return p1xy
  p1xy = make_p1xy()
  _plot_surface(p1xy)

In [5]:
if __name__ == '__main__' and '__file__' not in globals():
  p1y = _marginalize(p1xy, 'y')
  _plot_function(p1y)

In [6]:
if __name__ == '__main__' and '__file__' not in globals():
  p1x = _marginalize(p1xy, 'x')
  _plot_function(p1x)

In [7]:
def _simpson_integrate_2d(f_xy, ax = -1024, bx = 1024, ay = -1024, by = 1024):
  def inner(x): # ∫ f_xy(x, y) dy
    return _simpson_integrate(lambda y: f_xy(x, y), ay, by)
  return _simpson_integrate(inner, ax, bx) # ∫ inner(x) dx

In [8]:
def _find_expected_value_2d(f_xy, p_xy):
  return _simpson_integrate_2d(lambda x, y: f_xy(x, y) * p_xy(x, y))

In [10]:
if __name__ == '__main__' and '__file__' not in globals():
  lin_iden_1d = lambda x: x
  lin_iden_2d = lambda x, y: x * y

  e1xy = _find_expected_value_2d(lin_iden_2d, p1xy)
  print(e1xy)

  e1x = _find_expected_value(lin_iden_1d, p1x)
  print(e1x)

  e1y = _find_expected_value(lin_iden_1d, p1y)
  print(e1y)

-7.501369255020594e-18
-1.6761111524513806e-17
1.9999999964329498
